# Display Nonlinearity — How does the monitor interpret the values we send?

In this notebook we explore how a monitor's nonlinear response (gamma) affects image brightness,
and how the sRGB standard describes this nonlinearity mathematically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Ignore this code. It will be needed for 1:1 display in jupyther notebooks
from one_to_one_drawing import (
    build_guide_rgb,
    copy_array_into_rgb_buffer,
    show_rgb_buffer_plotly_strict,
)

# ---- Display-target parameters ----
DISPLAY_WIDTH = 1024
DISPLAY_HEIGHT = 512  # informational only
WINDOW_WIDTH = 1200    # plenty big enough to show the arrow and the red guards on either side
WINDOW_HEIGHT = 400    # tall enough not just for arrow but for a single mosaic strip
ARROW_HEIGHT = 11      # odd integer >= 3
RED_GUARD_WIDTH = 2    # 2 px full-height columns immediately beyond each tip

# If True, use 0/255 uint8 image; if False, keep float 0..1 then convert near the end.
USE_UINT8 = True

## Part 1 — Mixing brightness from black and white pixels

A monitor mixes perceived brightness physically by alternating black and white pixels —
similar to how an inkjet printer works.

The image below shows checkerboard patterns at increasing tile sizes.
**View this image at 100% zoom (1:1 pixels).** Step back from your monitor:
all checkerboard fields should appear as the same grey regardless of tile size.

## Part 2 — Match a solid grey to a 50% checkerboard

The image below shows (left to right): **black**, a **50% checkerboard**, and **white**.
Below that is a solid grey patch whose brightness is controlled by `test_color`.

Adjust `test_color` until both grey fields look equally bright.
Why is the matching value not 0.5?

In [ ]:
s = 64
test_color = 0.2140   # <-- adjust this until top and bottom grey look identical

checker = np.tile([[0, 1], [1, 0]], (s//2, s//2)).astype(float)
row_top = np.hstack([np.zeros((s, s)), checker, np.ones((s, s))])
row_bot = np.hstack([np.zeros((s, s)), np.full((s, s), test_color), np.ones((s, s))])
img_both = np.vstack([row_top, row_bot])

fig, ax = plt.subplots(figsize=(6, 2), dpi=100)
ax.imshow(img_both, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
ax.set_title(f'Top: 50% checker   Bottom: solid grey = {test_color:.4f}')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
MOSAIC_SIZE = 128

def make_tile(chip_size: int) -> np.ndarray:
    """Make a 2x2 tile with the individual chips being of the given size."""
    m00 = np.zeros((chip_size, chip_size), dtype=np.uint8)
    m01 = np.ones((chip_size, chip_size), dtype=np.uint8)
    m10 = np.ones((chip_size, chip_size), dtype=np.uint8)
    m11 = np.zeros((chip_size, chip_size), dtype=np.uint8)
    top = np.hstack((m00, m01))
    bottom = np.hstack((m10, m11))
    return np.vstack((top, bottom))

def make_mosaic(chip_size: int) -> np.ndarray:
    """Make a 2x2 mosaic of the given tile."""
    tile = make_tile(chip_size)
    return np.tile(tile, (MOSAIC_SIZE // (2 * chip_size), MOSAIC_SIZE // (2 * chip_size)))

def make_mosaic_strip():
    mosaics = [make_mosaic(2**i) for i in range(7)]
    return np.hstack(mosaics)

mosaic_strip = make_mosaic_strip()

In [ ]:
def plot_with_inserted_array(inserted_array: np.ndarray) -> None:

    rgb, safe_roi = build_guide_rgb(
        window_w=WINDOW_WIDTH,
        window_h=WINDOW_HEIGHT,
        arrow_w=DISPLAY_WIDTH,
        arrow_h=ARROW_HEIGHT,
        red_guard_w=RED_GUARD_WIDTH,
    )
    safe_x_begin, safe_x_end, safe_y_begin, safe_y_end = safe_roi

    ins_x_begin = safe_x_begin
    ins_x_end = ins_x_begin + inserted_array.shape[1]
    if ins_x_end > safe_x_end:
        raise ValueError("Inserted array is too wide to fit in the safe ROI.")
    ins_y_begin = safe_y_begin
    ins_y_end = ins_y_begin + inserted_array.shape[0]
    if ins_y_end > safe_y_end:
        raise ValueError("Inserted array is too tall to fit in the safe ROI.")

    copy_array_into_rgb_buffer(
        rgb_buffer=rgb,
        array_to_copy=inserted_array,
        x_begin=ins_x_begin,
        y_begin=ins_y_begin,
    )

    fig = show_rgb_buffer_plotly_strict(rgb)

In [ ]:
plot_with_inserted_array(mosaic_strip)

## Part 3 — Build a full grey ramp and measure monitor nonlinearity

We create dithered patches for a range of physical grey levels
(0, 1/64, 2/64, 1/16 … 3/4, 1) and match each one with a solid grey value.
The resulting curve reveals the monitor's nonlinearity.

In [ ]:
s = 64

# Helper: create a dithered tile and upscale to s x s
def make_patch(tile):
    n = tile.shape[0]
    return np.tile(tile, (s // n, s // n)).astype(float)

# Dithered tiles for each physical grey level
patches = [
    make_patch(np.zeros((1, 1))),                               # 0
    make_patch(np.array([[1,0,0,0,0,0,0,0],                    # 1/64
                          [0,0,0,0,0,0,0,0]] * 4).reshape(8,8)),
    make_patch(np.array([[1,0,0,0,0,0,0,0],                    # 2/64
                          [0,0,0,0,1,0,0,0]] * 4).reshape(8,8)),
    make_patch(np.array([[1,0,0,0],[0,0,0,0],
                          [0,0,0,0],[0,0,0,0]])),               # 1/16
    make_patch(np.array([[1,0,0,0],[0,0,0,0],
                          [0,0,1,0],[0,0,0,0]])),               # 2/16
    make_patch(np.array([[1,0,0,0],[0,0,1,0],
                          [0,0,0,0],[0,0,1,0]])),               # 3/16
    make_patch(np.array([[0,1],[1,0]])),                        # 1/4
    make_patch(np.array([[1,0,1,0],[0,1,0,0],
                          [1,0,1,0],[0,0,0,1]])),               # 6/16
    make_patch(np.array([[1,0],[0,1]])),                        # 2/4 = 0.5
    make_patch(np.array([[1,1],[0,1]])),                        # 3/4
    make_patch(np.ones((1, 1))),                                # 1
]

ramp_top = np.hstack(patches)

# Matched solid grey values (adjust these by eye until rows look equal)
test_colors = np.array([0.0, 0.11155, 0.19412, 0.27431, 0.37682,
                         0.45265, 0.52198, 0.62744, 0.71707, 0.85632, 1.0])

ramp_bot = np.hstack([np.full((s, s), v) for v in test_colors])
img_ramp = np.vstack([ramp_top, ramp_bot])

fig, ax = plt.subplots(figsize=(12, 2), dpi=100)
ax.imshow(img_ramp, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
ax.set_title('Top: dithered patches   Bottom: matched solid greys')
ax.axis('off')
plt.tight_layout()
plt.show()

## Part 4 — Plot measured curve vs. sRGB standard

The sRGB nonlinearity is defined as:

$$L_{\text{linear}} = \begin{cases}
\left(\frac{V + 0.055}{1.055}\right)^{2.4} & \text{if } V > 0.04045 \\
\frac{V}{12.92} & \text{otherwise}
\end{cases}$$

where $V$ is the encoded signal value (0…1) and $L$ is the linear light output.

In [ ]:
# sRGB transfer functions
sRGB2linear = lambda x: np.where(x > 0.04045, ((x + 0.055) / 1.055) ** 2.4, x / 12.92)
linear2sRGB = lambda x: np.where(x > 0.0031308, 1.055 * x ** (1/2.4) - 0.055, 12.92 * x)

ref_colors = np.array([0, 1/64, 2/64, 1/16, 2/16, 3/16, 1/4, 6/16, 2/4, 3/4, 1.0])

x = np.linspace(0, 1, 500)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(test_colors, ref_colors, 'o-', lw=2, label='Measured (color matching)')
ax.plot(x, sRGB2linear(x), ':', lw=2, label='sRGB standard')
ax.set_xlabel('Monitor input value (encoded)')
ax.set_ylabel('Linear light output')
ax.set_title('Monitor nonlinearity')
ax.legend()
ax.grid(True)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Part 5 — Effect of monitor nonlinearity on images

We download a test image encoded in sRGB and show it:
1. As-is (correct for a standard sRGB monitor)
2. Linearized and then re-encoded for the measured monitor curve

Toggle between the two to see the difference.

In [ ]:
import urllib.request
from PIL import Image
import io

url = 'https://cdn.shopify.com/s/files/1/2691/1130/files/ARRIColorTool_grande.jpg?v=1524533887'
with urllib.request.urlopen(url) as r:
    img_sRGB = np.array(Image.open(io.BytesIO(r.read())).convert('RGB')) / 255.0

img_sRGB = img_sRGB[60:240, 360:, :]

# Linearize, then re-encode with the measured curve
img_lin = sRGB2linear(img_sRGB)
img_monitor = np.clip(
    np.interp(img_lin.ravel(), ref_colors, test_colors).reshape(img_lin.shape), 0, 1
)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(img_sRGB)
axes[0].set_title('Standard sRGB display')
axes[0].axis('off')
axes[1].imshow(img_monitor)
axes[1].set_title('Corrected for measured monitor curve')
axes[1].axis('off')
plt.tight_layout()
plt.show()